In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

#root_path = Path().resolve().parent

load_dotenv(".env")

TMDB_TOKEN = os.getenv("TMDB_TOKEN")


In [ ]:
import os
import time
import requests
import pandas as pd
from pytrends.request import TrendReq

# ── Konfiguration ─────────────────────────────────────

TOTAL_MOVIES = 120
PAGES = 6            # 20 movies per page
GEO = "DE"

OUTPUT_FILE = os.path.join(os.getcwd(), "Q9.csv")

tmdb_headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

pytrends = TrendReq(hl="de-DE", tz=60)

all_rows = []

# ── Step 1: Filme automatisch laden ───────────────────

movies = []

for page in range(1, PAGES + 1):

    resp = requests.get(
        "https://api.themoviedb.org/3/discover/movie",
        headers=tmdb_headers,
        params={
            "sort_by": "popularity.desc",
            "vote_count.gte": 500,
            "primary_release_date.gte": "2010-01-01",
            "primary_release_date.lte": "2023-12-31",
            "page": page
        }
    ).json()

    movies.extend(resp["results"])

movies = movies[:TOTAL_MOVIES]

print(f"Loaded {len(movies)} movies from TMDB")


# ── Step 2: Daten pro Film sammeln ────────────────────

for movie in movies:

    title = movie["title"]
    release = pd.to_datetime(movie["release_date"])

    trends_start = (release - pd.DateOffset(months=3)).strftime("%Y-%m-%d")
    trends_end = release.strftime("%Y-%m-%d")

    print(f"Processing: {title}")

    # Movie Details
    details = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie['id']}",
        headers=tmdb_headers
    ).json()

    budget = details.get("budget")
    revenue = details.get("revenue")

    if not budget or not revenue:
        print("  skipped (missing financial data)")
        continue

    # ── Google Trends ──
    try:
        pytrends.build_payload(
            kw_list=[title],
            timeframe=f"{trends_start} {trends_end}",
            geo=GEO
        )

        trends_df = pytrends.interest_over_time()

        avg_before = trends_df[title].mean() if not trends_df.empty else None
        max_before = trends_df[title].max() if not trends_df.empty else None
        peak_date = trends_df[title].idxmax() if not trends_df.empty else None

    except:
        avg_before = None
        max_before = None
        peak_date = None

    # ── Datensatz speichern ──

    all_rows.append({
        "title": title,
        "release_date": details.get("release_date"),
        "budget": budget,
        "revenue": revenue,
        "roi": round(revenue / budget, 2),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count"),
        "runtime": details.get("runtime"),
        "genres": ", ".join(g["name"] for g in details.get("genres", [])),
        "avg_trend_before": round(avg_before, 2) if avg_before else None,
        "max_trend_before": max_before,
        "trend_peak_date": peak_date,
    })

    time.sleep(1)   # wichtig wegen Google Trends Rate Limit


# ── Step 3: CSV speichern ─────────────────────────────

df = pd.DataFrame(all_rows)

df.to_csv(OUTPUT_FILE, index=False)

print("\nDataset erstellt")
print(f"Filme im Datensatz: {len(df)}")
print(f"Gespeichert unter: {OUTPUT_FILE}")
print(df.head())